# Lekse 11 - Agent-til-Agent (A2A) Protokoll


## Oppsett


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Hva er A2A-protokollen?

**Agent-til-Agent (A2A) protokollen** er en åpen standard som gjør det mulig for AI-agenter å kommunisere,
oppdage hverandre og samarbeide — selv når de er bygget på forskjellige rammeverk eller hostet
av forskjellige tjenester.

Nøkkelkonsepter:

- **Oppdagelse** – Agenter publiserer et *Agentkort* som beskriver deres kapasiteter, noe som gjør det
  lett for andre agenter (eller orkestratorer) å finne den rette spesialisten for en oppgave.
- **Meldingsutveksling** – Agenter utveksler strukturerte meldinger gjennom en felles protokoll, slik at en
  forespørsel fra en agent kan forstås og utføres av en annen uavhengig av dens interne
  implementering.
- **Oppgavelivssyklus** – A2A definerer tilstander som *innsendt*, *arbeider*, *fullført* og
  *feilet*, noe som gir orkestratoren full innsikt i hvordan en delegert oppgave utvikler seg.

I denne leksjonen simulerer vi A2A-stil samarbeid ved å koble tre spesialiserte reiseagenter
inn i en arbeidsflyt hvor hver agent bidrar med sin ekspertise og sender resultater videre til den neste.


## Lage spesialiserte reisebyråer


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Samhandling mellom flere agenter via arbeidsflyt

Vi kobler de tre agentene sammen i en sekvensiell arbeidsflyt som speiler A2A-meldingsutveksling:

1. **CurrencyExchangeAgent** mottar brukerforespørselen og gir veiledning om valuta.
2. **ActivityPlannerAgent** mottar den berikede konteksten og legger til aktivitetsanbefalinger.
3. **TravelManagerAgent** syntetiserer begge inngangene til en endelig reisebrief.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Forståelse av A2A i produksjon

I et produksjonsmiljø åpner A2A-protokollen for kraftige scenarioer på tvers av tjenester:

| Mulighet | Beskrivelse |
|---|---|
| **Tverr-rammeverks interoperabilitet** | En agent bygget med ett rammeverk kan delegere oppgaver til en agent bygget med et annet A2A-kompatibelt rammeverk, noe som muliggjør ekte interoperabilitet på tvers av organisasjoner. |
| **Tjenestegrenser** | Agenter kan være plassert i separate mikrotjenester, skyløsninger eller til og med ulike organisasjoner, samtidig som de samarbeider sømløst. |
| **Dynamisk oppdagelse** | En orkestrator kan slå opp i et Agent Card-register under kjøring for å finne den mest egnede spesialisten for en gitt deloppgave. |
| **Strømming og push-varsler** | A2A støtter Server-Sent Events (SSE) for sanntids fremdriftsoppdateringer og push-varsler for langvarige oppgaver. |

Arbeidsflyten vi bygde over er en forenklet, in-prosess versjon av dette mønsteret. I en ekte
distribusjon ville hver agent eksponere et HTTP-endepunkt, publisere et Agent Card, og kommunisere
via A2A JSON-RPC-protokollen.


## Sammendrag

I denne leksjonen lærte du:

1. **Hva A2A-protokollen er** — en åpen standard for oppdagelse, meldingstjenester,
   og oppgavehåndtering mellom agenter.
2. **Hvordan lage spesialiserte agenter** — en Valutaomvekslingsagent, en Aktivitetsplanlegger-agent,
   og en Reiseleder-orchestrator.
3. **Hvordan koble agenter inn i en arbeidsflyt** — ved å bruke `WorkflowBuilder` for å modellere sekvensiell
   meldingsoverføring mellom agenter.
4. **Hvordan A2A fungerer i produksjon** — som muliggjør samarbeid på tvers av rammeverk og tjenester
   med dynamisk oppdagelse og strømmende oppdateringer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
